# Fine-tune ViDeBERTa-xsmall — đọc gói và lưu kết quả trên Drive

Gói ZIP đã được đặt trong `My Drive/nlp_chuan_hoa_dia_chi/`. Chọn GPU rồi chạy từng mục; không dùng widget upload file. Mount Drive lần đầu có thể cần xác nhận tài khoản.

**Cấu hình mặc định:** 5 epoch, 4.841 mẫu train nhận diện và 1.169 mẫu riêng cho ranker. Không có nhãn test cuối. Checkpoint và báo cáo được lưu trên Drive. Train GPU dự trù 10–30 phút, ETA thực được in sau 10 batch.

## 1. Kiểm tra GPU

In [ ]:
import sys, subprocess, torch
assert torch.cuda.is_available(), "Bật GPU ở Runtime → Change runtime type trước khi chạy tiếp"
print("Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name())
subprocess.run(["nvidia-smi"], check=True)


## 2. Đọc gói từ Drive
Kết nối Drive rồi giải nén gói đã có vào `/content` để xử lý nhanh. Kiểm tra SHA-256 trước khi dùng. Không cần upload lại ZIP khi Colab bị ngắt.

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, os, sys, hashlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/nlp_chuan_hoa_dia_chi")
ZIP_PATH = DRIVE_ROOT / "nlp_dia_chi_colab_v1.zip"
assert ZIP_PATH.is_file(), f"Chưa thấy ZIP đồng bộ lên Drive: {ZIP_PATH}"
assert hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest() == "e346d0b800a315c78fbdf9c705dd2a80144dbeeae5b2427793f54f40d840bd1f", "ZIP khác bản đã kiểm tra"
PROJECT = Path("/content/nlp_dia_chi_colab")
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in z.namelist():
        assert (Path("/content") / name).resolve().is_relative_to(PROJECT), name
    z.extractall("/content")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Đã đọc gói từ Drive:", ZIP_PATH)


## 3. Cài thư viện
Giữ nguyên PyTorch CUDA của Colab. Không chạy `requirements-neural.txt` vì đó là môi trường CPU cũ.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "colab/requirements.txt"], check=True)
assert torch.cuda.is_available(), "CUDA phải còn hoạt động sau cài thư viện"


## 4. Kiểm tra gói và smoke test
Các test dùng backbone tí hon để kiểm tra gradient, lưu/nạp, Unicode và pipeline; chưa phải fine-tune đầy đủ.

In [ ]:
subprocess.run([sys.executable, "-m", "colab.verify_bundle", "--root", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "unittest", "tests.test_videberta_colab", "tests.test_char", "tests.test_pipeline", "tests.test_pipeline_v3", "tests.test_crf", "tests.test_danh_gia"], check=True)


## 5. Nơi lưu và cấu hình
Drive được khuyên dùng để giữ checkpoint. Nếu không dùng Drive, cần tải kết quả về trước khi Colab kết thúc. Giữ nguyên cấu hình khi resume. Muốn đổi batch/epoch thì đổi RUN_ID để tạo một lần thử mới. Nếu hết VRAM, dùng BATCH_SIZE=4 và GRAD_ACC=4 trong một run mới.

In [ ]:
USE_DRIVE = True
RUN_ID = "videberta_v1_01"
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACC = 2
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/nlp_chuan_hoa_dia_chi") / RUN_ID
else:
    RUN_DIR = Path("/content/runs") / RUN_ID
print("Run:", RUN_DIR, "Resume:", RUN_DIR.exists())


## 6. Fine-tune GPU
Checkpoint pretrained được tải trực tiếp từ commit đã chốt. Lưu model và resume sau mỗi epoch. Resume quay lại epoch hoàn chỉnh gần nhất; nếu chưa xong epoch nào thì bắt đầu lại epoch 1. Không dùng dev để chọn checkpoint.

In [ ]:
cmd = [sys.executable, "-u", "-m", "colab.train_videberta", "--data-root", "bundle_data", "--run-dir", str(RUN_DIR), "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--gradient-accumulation", str(GRAD_ACC)]
if RUN_DIR.exists():
    cmd.append("--resume")
subprocess.run(cmd, check=True)
print((RUN_DIR / "training.json").read_text())


## 7. Train ranker và đánh giá CPU
Ranker học trên phần train riêng; đo dev với luật v3, CRF cũ, ViDeBERTa dùng điểm tay và dùng ranker. **Độ trễ ở đây là CPU Colab**, không phải CPU laptop. Tỷ lệ khớp là với nhãn máy/tổng hợp, chưa phải gold. Muốn đo lại, đổi tên thư mục báo cáo để giữ lịch sử.

In [ ]:
REPORT_DIR = RUN_DIR / "reports_cpu_colab_01"
subprocess.run([sys.executable, "-u", "-m", "colab.evaluate_videberta", "--run-dir", str(RUN_DIR), "--output", str(REPORT_DIR)], check=True)
print((REPORT_DIR / "BAO_CAO.md").read_text())


## 8. Thử một địa chỉ trên CPU

In [ ]:
subprocess.run([sys.executable, "-m", "src", "Số 12, Đường Lê Lợi, Phường Trúc Bạch, Quận Ba Đình, Hà Nội", "--videberta-model", str(RUN_DIR / "model"), "--ranker", str(RUN_DIR / "ranker.txt"), "--threads", "1"], check=True)


## 9. Xuất kết quả thẳng về Drive
ZIP kết quả được lưu trong `My Drive/nlp_chuan_hoa_dia_chi/ket_qua/`. Drive for desktop sẽ đồng bộ về máy; không cần widget download. Optimizer/resume lớn vẫn nằm trong thư mục run để tiếp tục khi bị ngắt.

In [ ]:
from datetime import datetime
EXPORT_DIR = DRIVE_ROOT / "ket_qua"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_ZIP = EXPORT_DIR / (RUN_ID + "_ket_qua_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip")
subprocess.run([sys.executable, "-m", "colab.export_results", "--run-dir", str(RUN_DIR), "--output", str(RESULT_ZIP)], check=True)
print("Đã lưu kết quả trên Drive:", RESULT_ZIP)
